# Safety & Governance Module — NYC 311 Data Exploration

**Author:** Arbaz Khalid — Safety & Governance Module  
**Project:** Plexa — Multi-Modal Smart City Digital Twin and Cascading Resilience Engine  
**Status:** Exploration — checking what NYC 311 data can give the Safety and Governance parts

## Overview

This notebook is part of the Safety & Citizen Governance module of Plexa. While the Climate module relates aggregate rainfall to citywide complaint volumes, this module works on individual citizen reports: Safety decides whether a report is a real hazard, and Governance extracts what and where.

## Purpose (exploration only)

1. Query the NYC 311 Socrata Open Data API for the Hurricane Ida window (Aug 28 – Sep 3, 2021).
2. List the available fields and complaint types.
3. Check whether any field holds real citizen free text.
4. Check how many reports carry coordinates.
5. Make a first draft of the hazard labels (flood, power_outage, road_blockage, non_hazard) and look at the class balance.

Cleaning, final labelling and the incident node export are done in later notebooks.

In [2]:
import os
import requests
import pandas as pd

URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"   # NYC 311 Service Requests
START = "2021-08-28T00:00:00"
END   = "2021-09-03T00:00:00"
WHERE = f"created_date between '{START}' and '{END}'"

def soda_get(params):
r = requests.get(URL, params=params, timeout=60)
r.raise_for_status()
return r.json()

In [3]:
sample = pd.DataFrame(soda_get({"$where": WHERE, "$limit": 20}))
print(f"{sample.shape[1]} columns")
print(list(sample.columns))
sample.head(3).T

36 columns
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'status', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'latitude', 'longitude', 'location', 'descriptor_2', 'facility_type']


,0,1,2
unique_key,51734887,51742446,51731090
created_date,2021-09-03T00:00:00.000,2021-09-03T00:00:00.000,2021-09-03T00:00:00.000
closed_date,2021-09-09T00:00:01.000,2021-09-09T00:00:01.000,2021-09-07T00:00:01.000
agency,DOHMH,DOHMH,DOHMH
agency_name,Department of Health and Mental Hygiene,Department of Health and Mental Hygiene,Department of Health and Mental Hygiene
complaint_type,Food Poisoning,Food Poisoning,Food Poisoning
descriptor,1 or 2,1 or 2,1 or 2
location_type,Restaurant/Bar/Deli/Bakery,Restaurant/Bar/Deli/Bakery,Restaurant/Bar/Deli/Bakery
incident_zip,10001,11418,11238
incident_address,224 WEST 35 STREET,104-02 JAMAICA AVENUE,771 WASHINGTON AVENUE


## Step 1 — Fields

Pulled 20 sample reports to list all available columns and check for a free-text field.